# Skin Lesion Classification using CNN (V2 — Improved)

## Semester Project – Programming for AI

### Team Members
- bscs24043 Ibrahim Butt
- bscs24095 Waize Arif
- bscs24139 Syed Jaffar Raza Kazmi
- bscs24083 Muhammad Moiz

### Objective
Build a Convolutional Neural Network (CNN) model to classify skin lesion images.

### Changes from V1
- Stratified train/test split (balanced class distribution)
- Learning rate scheduler (ReduceLROnPlateau)
- Early stopping with best model saving
- Test-Time Augmentation (TTA) for evaluation
- More epochs (80) with generous patience
- Per-class accuracy reporting and training curves

# Python Imports

In [ ]:
!pip install torchmetrics
!pip install torchvision
!pip install scikit-learn

import torch
import torchvision
from torch.utils.data import DataLoader, Subset
import torchmetrics
from torchmetrics import ConfusionMatrix
import matplotlib.pyplot as plt
import numpy as np
from torch import nn
from timeit import default_timer as timer
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from sklearn.model_selection import train_test_split


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp "/content/drive/MyDrive/skin_lesion/skin_dataset.zip" "/content/"

!unzip -q "/content/skin_dataset.zip" -d "/content/"

!ls "/content/dataset"

# Check Version and Device

In [ ]:
print(f"PyTorchColab Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# HyperParameters

In [ ]:
# --- HYPERPARAMETER DASHBOARD ---

SEED = 42                # seed for reproducibility
TRAIN_SPLIT = 0.8        # train test split value, 80% train, 20% test
BATCH_SIZE = 32          # same as V1
LEARNING_RATE = 0.0005   # same as V1
EPOCHS = 80              # more room, early stopping handles the rest
DROPOUT_RATE = 0.3       # same as V1
IMAGE_SIZE = 224         # same as V1
PATIENCE = 15            # generous patience for small dataset

# Transforming Data

In [ ]:
# DATA TRANSFORMING — same as V1 (proven to work)

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomVerticalFlip(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])
print("Transforms defined successfully.")

# Data Loading

**V2 Change:** Stratified split ensures every class has proportional representation in both train and test sets. With random split and small data, some classes can end up underrepresented.

In [ ]:
dataset_path = "/content/dataset"

train_data_full = ImageFolder(root=dataset_path, transform=train_transforms)
test_data_full  = ImageFolder(root=dataset_path, transform=test_transforms)

# STRATIFIED split: ensures each class is proportionally represented
# V1 used random split which could leave some classes underrepresented
all_targets = train_data_full.targets
all_indices = list(range(len(train_data_full)))

train_indices, test_indices = train_test_split(
    all_indices,
    test_size=(1 - TRAIN_SPLIT),
    stratify=all_targets,
    random_state=SEED
)

# train and test sets
train_dataset = Subset(train_data_full, train_indices)
test_dataset  = Subset(test_data_full, test_indices)

# Loaders made
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Show per-class distribution
print(f"Classes: {train_data_full.classes}")
print(f"Total images: {len(train_data_full)}")
print(f"Training: {len(train_dataset)} | Testing: {len(test_dataset)}")

# Verify stratification
from collections import Counter
train_class_counts = Counter([all_targets[i] for i in train_indices])
test_class_counts = Counter([all_targets[i] for i in test_indices])
print(f"\nTrain distribution: {dict(sorted(train_class_counts.items()))}")
print(f"Test distribution:  {dict(sorted(test_class_counts.items()))}")
print("Data loaded correctly. Train and Test are stratified and separated.")

# The CNN Class

Same architecture as V1 — it already achieved 77%.

In [ ]:
class SkinCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()

        self.dropout = nn.Dropout(DROPOUT_RATE)

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # 128 channels * 28 height * 28 width
        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.fc2 = nn.Linear(256, 5)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))

        x = self.flatten(x)

        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = SkinCNN().to(device)
criterion = nn.CrossEntropyLoss()

# Same optimizer as V1 (no weight_decay — let the big FC learn freely)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# LR scheduler: halves LR when loss stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=4, factor=0.5
)

print("CNN built and pushed to GPU.")


# Training Loop

In [ ]:
train_loss_history = []
test_acc_history = []
best_accuracy = 0.0
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    train_loss_history.append(epoch_loss)

    # ---- TESTING (with TTA) ----
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Test-Time Augmentation: average predictions from
            # original + horizontally flipped + vertically flipped
            out_original = model(inputs)
            out_hflip    = model(torch.flip(inputs, dims=[3]))
            out_vflip    = model(torch.flip(inputs, dims=[2]))

            outputs = (out_original + out_hflip + out_vflip) / 3.0

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total
    test_acc_history.append(epoch_acc)

    # Step the scheduler
    scheduler.step(epoch_loss)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Test Acc: {epoch_acc:.2f}% | LR: {current_lr:.6f}")

    # Save best model
    if epoch_acc > best_accuracy:
        best_accuracy = epoch_acc
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        print(f"  -> New best accuracy! Saving model.")
    else:
        patience_counter += 1

    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered after {epoch+1} epochs (no improvement for {PATIENCE} epochs).")
        break

# Restore best model
model.load_state_dict(best_model_state)
print(f"\nTraining complete. Best Test Accuracy: {best_accuracy:.2f}%")

# Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(range(1, len(train_loss_history)+1), train_loss_history, 'b-', linewidth=2)
ax1.set_title('Training Loss per Epoch', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2.plot(range(1, len(test_acc_history)+1), test_acc_history, 'g-', linewidth=2)
ax2.set_title('Test Accuracy per Epoch', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Accuracy and Prediction Checking

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)

        # TTA for final evaluation too
        out_original = model(inputs)
        out_hflip    = model(torch.flip(inputs, dims=[3]))
        out_vflip    = model(torch.flip(inputs, dims=[2]))
        outputs = (out_original + out_hflip + out_vflip) / 3.0

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Per-class accuracy report
print("Per-Class Classification Report:")
print("=" * 60)
print(classification_report(all_labels, all_preds, target_names=train_data_full.classes))

# confusion matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_data_full.classes)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
plt.title("Confusion Matrix: Actual vs Predicted")
plt.show()